# OpenAI + LangChain

LangChain es un framework que nos permite crear agentes conversacionales de manera flexible y modular, integrando modelos de lenguaje como OpenAI GPT con herramientas, flujos de trabajo y memoria. Simplifica la gestión de agentes, herramientas y cadenas de prompts, ofreciendo la posibilidad de construir desde pequeños ejemplos locales hasta sistemas más complejos de producción.

Podemos levantar un entorno local siguiendo los pasos de instalación:

In [ ]:
#pip install langchain openai


In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.agents import initialize_agent, Tool
from langchain.memory import ConversationBufferMemory

llm = ChatOpenAI(model_name="gpt-4", temperature=0)

def calculadora(texto: str) -> str:
    return str(eval(texto)) 

tools = [
    Tool(
        name="Calculadora",
        func=calculadora,
        description="Realiza cálculos matemáticos simples"
    )
]

# Crear memoria para el historial de la conversación
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# Inicializar agente conversacional
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent="conversational-react-description",  # decide automáticamente cuándo usar herramientas
    memory=memory,
    verbose=True
)

respuesta = agent.run("Hola, ¿puedes ayudarme con unos cálculos?")
print(respuesta)


In [ ]:
agent.run("¿Cuánto es 4 * 5?")

In [ ]:
import os
from opentelemetry import trace as trace_api
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter

from langchain.chat_models import ChatOpenAI
from langchain.schema import HumanMessage

# 1. Variables de entorno
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY", "")

# 2. Configurar OTLP exporter (opcional)
endpoint = "https://api.smith.langchain.com/otel/v1/traces"
headers = {
    "x-api-key": os.getenv("LANGSMITH_API_KEY"),
    "Langsmith-Project": os.getenv("LANGSMITH_PROJECT"),
}
tracer_provider = TracerProvider()   # envía los datos de trazas (cada interacción) al endpoint de LangSmith. Esto permite visualizar en tiempo real las llamadas al LLM y sus herramientas.
tracer_provider.add_span_processor(   # https://smith.langchain.com/o/74ea8a72-ba54-4355-a21f-2c6b3c2409dc/projects
    SimpleSpanProcessor(OTLPSpanExporter(endpoint=endpoint, headers=headers))
)
trace_api.set_tracer_provider(tracer_provider)

# 3. Definir LLM
llm = ChatOpenAI(model_name="gpt-4", temperature=0)

# 4. Ejecutar una llamada usando HumanMessage
mensaje = HumanMessage(content="¿Cuánto es 4 * 5?")
response = llm([mensaje])

# 5. Imprimir respuesta
print(response.content)



Podemos extender el uso de herramientas tal y como vimos con LangChain.

In [ ]:
def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

In [ ]:
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# --- Memoria de la conversación ---
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# --- LLM con streaming ---
llm = ChatOpenAI(
    model_name="gpt-4",
    temperature=0,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

# --- Agente conversacional ---
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent="conversational-react-description",
    memory=memory,
    verbose=True
)

# --- Uso del agente ---
agent.run("¿Cuánto es 4 * 5?")

Podemos incluir herramientas más elaboradas como los buscadores web.

In [ ]:
#! pip install tavily-python

In [ ]:
import json 
from openai import OpenAI 
from tavily import TavilyClient

client = OpenAI()

tavily = TavilyClient(api_key="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")

# --- HERRAMIENTA ---

def tavily_search(query: str) -> str:
    """
    Busca información actualizada en la web sobre viajes, destinos y rutas. 
    Útil para encontrar opciones de vuelo, tren o autobús.
    """
    results = tavily.search(query=query, search_depth="basic")
    context = "\n".join([result['content'] for result in results['results']])
    return context


# --- PROMPT DEL SISTEMA ---

SYSTEM_PROMPT = """
    Eres un especialista en viajes trabajando en una agencia. 
    Tu trabajo es proponer opciones de viajes a los clientes aunque tienes una predilección 
    por Bilbao y siempre que puedes lo propones como destino.
    Debes utilizar la herramienta de búsqueda web cuando necesites información 
    actualizada sobre rutas, horarios o precios.
"""

# --- FUNCIÓN PRINCIPAL DEL AGENTE ---

def travel_agent_openai(user_query: str):
    available_tools = {
        "tavily_search": tavily_search
    }

    tools_spec = [
        {
            "type": "function",
            "function": {
                "name": "tavily_search",
                "description": available_tools['tavily_search'].__doc__.strip(),
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "La consulta a buscar en la web."},
                    },
                    "required": ["query"],
                },
            }
        }
    ]

    response = client.chat.completions.create(
        model="gpt-4o", 
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_query}
        ],
        tools=tools_spec,
        tool_choice="auto" 
    )

    first_response_message = response.choices[0].message
    
    if first_response_message.tool_calls:
        print("🤖 Agente: Necesito usar la herramienta de búsqueda web...")
        tool_calls = first_response_message.tool_calls
        
        tool_outputs = []
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_to_call = available_tools.get(function_name)
            
    
            function_args = json.loads(tool_call.function.arguments)
            
            function_response = function_to_call(query=function_args.get("query"))
            
            tool_outputs.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "content": function_response, 
            })
            
        print("🤖 Agente: Re-consultando a GPT-4o con los resultados...")
        
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_query},
            first_response_message, 
            *tool_outputs           
        ]

        final_response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
        )
        
        print("\n**✨ Respuesta Final del Agente (GPT-4o) ✨**")
        print(final_response.choices[0].message.content)

    else:
        print("\n**✨ Respuesta Final del Agente (GPT-4o) - Sin herramienta ✨**")
        print(first_response_message.content)


# --- EJECUCIÓN ---

travel_agent_openai(
    "¿Qué viajes hay a Madrid desde Santander?"
)

In [ ]:
travel_agent_openai(
    "Está bien, busca entonces vuelos a ese destino"
)

En muchos casos necesitaremos que nuestro interlocutor nos de el ok a la operación. Para eso, debemos instruir al agente de que la ejecución de la herramienta debe disponer de una aceptación.

In [ ]:
import json
from typing import Any, Callable, Dict, Iterator, List, Union
import httpx
from rich.console import Console
from rich.prompt import Prompt

class ToolCallCancelled(Exception):
    """Excepción para indicar que la llamada a la herramienta fue cancelada por el usuario."""
    pass

console = Console()

def confirmation_hook(
    function_name: str, function_call: Callable, arguments: Dict[str, Any]
) -> Any:
    """
    Función que solicita confirmación del usuario antes de ejecutar una herramienta.
    
    Esta función NO se usa directamente como un decorador en la librería OpenAI.
    Debe ser llamada manualmente ANTES de ejecutar la función de herramienta
    después de que el modelo de OpenAI la haya solicitado.
    """
    
    console.print(f"\nVoy a ejecutar [bold blue]{function_name}[/]")
    message = (
        Prompt.ask("¿Quieres que proceda?", choices=["s", "n"], default="s")
        .strip()
        .lower()
    )

    if message != "s":
        raise ToolCallCancelled(
            "Tool call cancelled by user. Stopping execution as permission was not granted."
        )
    
    result = function_call(**arguments)

    return result

def get_top_hackernews_stories(num_stories: int) -> str:
    """Fetch top stories from Hacker News and return a JSON string.

    Args:
        num_stories (int): Number of stories to retrieve

    Returns:
        str: JSON string containing story details
    """
    response = httpx.get("https://hacker-news.firebaseio.com/v0/topstories.json")
    response.raise_for_status() 
    story_ids = response.json()

    # Get story details
    final_stories = []
    for story_id in story_ids[:num_stories]:
        story_response = httpx.get(
            f"https://hacker-news.firebaseio.com/v0/item/{story_id}.json"
        )
        story_response.raise_for_status() 
        story = story_response.json()
        
        if "text" in story:
            story.pop("text", None)
            
        final_stories.append(story)

    # Devolver un JSON string
    return json.dumps(final_stories)

# --- Definición de la herramienta para OpenAI ---
# Esta es la parte que la librería OpenAI necesita para usar la función como herramienta.
# La integración del confirmation_hook debe hacerse ANTES de llamar a esta función.
TOOLS_FOR_OPENAI = [
    {
        "type": "function",
        "function": {
            "name": get_top_hackernews_stories.__name__,
            "description": get_top_hackernews_stories.__doc__.split('\n')[0], 
            "parameters": {
                "type": "object",
                "properties": {
                    "num_stories": {
                        "type": "integer",
                        "description": "Number of stories to retrieve.",
                    }
                },
                "required": ["num_stories"],
            },
        },
    }
]

# --- Diccionario de funciones para llamar a la herramienta ---
# Útil para mapear el nombre de la función que solicita el modelo de OpenAI
# a la función Python real.
AVAILABLE_TOOLS: Dict[str, Callable] = {
    "get_top_hackernews_stories": get_top_hackernews_stories,
}

In [ ]:
import os
from openai import OpenAI

client = OpenAI() 

SYSTEM_INSTRUCTIONS = "Eres un especialista en periodismo tecnológico y puedes predecir tendencias de mercado basado en noticias de hackernews."

USER_PROMPT = "¿Qué disrupciones prevés para este final de año?"

response = client.chat.completions.create(
    model="gpt-4o-mini", 
    messages=[
        {"role": "system", "content": SYSTEM_INSTRUCTIONS},
        {"role": "user", "content": USER_PROMPT},
    ],
    tools=TOOLS_FOR_OPENAI, 
    tool_choice="auto", 
    stream=True 
)


print(f"\n[bold green]Respuesta del Modelo (Stream):[/]")
full_response = ""
for chunk in response:
    content = chunk.choices[0].delta.content
    tool_calls = chunk.choices[0].delta.tool_calls
    
    if content:
        console.print(content, end="")
        full_response += content
        
    if tool_calls:
    
    
        print("\n\n[bold yellow]¡El modelo ha solicitado una herramienta![/]")
        print("En este punto, deberías llamar a 'confirmation_hook' antes de ejecutarla.")
        break 